# Comparing forecasting models

Which model should you use to forecast your time series: machine learning, a statistical model, a foundation model or a neural network? This guide answers the question with data. The main model families available in skforecast are evaluated with **the same backtesting on three datasets** with different characteristics:

| Family | Model | Forecaster |
|:--|:--|:--|
| Baseline | Seasonal naive | [`ForecasterEquivalentDate`](../api/forecasterequivalentdate.html) |
| Machine learning (linear) | Ridge | [`ForecasterRecursive`](../api/forecasterrecursive.html), [`ForecasterRecursiveMultiSeries`](../api/forecasterrecursivemultiseries.html) |
| Machine learning (trees) | LightGBM | [`ForecasterRecursive`](../api/forecasterrecursive.html), [`ForecasterRecursiveMultiSeries`](../api/forecasterrecursivemultiseries.html) |
| Statistical | AutoARIMA | [`ForecasterStats`](../api/forecasterstats.html) |
| Foundation | Chronos-2 (zero-shot) | [`ForecasterFoundation`](../api/forecasterfoundation.html) |
| Deep learning | LSTM | [`ForecasterRnn`](../api/forecasterrnn.html) |

For each model, the guide reports the accuracy of the point forecast, the calibration of its 80% prediction interval and the computation time.

## Key takeaways

KEY_TAKEAWAYS_PLACEHOLDER

## How the models are compared

+ **Same evaluation**: in each dataset, all the models are evaluated on the same test period with the same [`TimeSeriesFold`](../api/model_selection.html#skforecast.model_selection._split.TimeSeriesFold). Each family has its own [backtesting](backtesting.ipynb) function, but the partitions are identical.

+ **No tuning on the test set**: the configuration of each model (lags, features, hyperparameters, network size) was chosen on a validation period before the test set. To keep the guide short, it is written here as fixed values. See [hyperparameter tuning](hyperparameter-tuning-and-lags-selection.ipynb) to search them.

+ **Prediction intervals of 80%**: each family uses its natural method. Conformal prediction with the out-of-sample residuals of a backtest before the test period for Ridge and LightGBM, conformal prediction with in-sample residuals for the seasonal naive and the LSTM, the parametric interval of ARIMA and the 0.1 and 0.9 quantiles of Chronos-2. See [probabilistic forecasting](probabilistic-forecasting-overview.md).

+ **Metrics**: MAE, MASE (the MAE divided by the MAE of the one-step naive forecast in the training set: lower is better and, being scale free, it can be compared across datasets and series), coverage of the interval (the closer to 80%, the better calibrated) and the time of the whole backtesting, on CPU.

In [ ]:
# Libraries
# ==============================================================================
import os
os.environ["KERAS_BACKEND"] = "torch"  # 'tensorflow', 'jax' or 'torch'
import time
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor

import skforecast
from skforecast.datasets import fetch_dataset, load_demo_dataset
from skforecast.recursive import (
    ForecasterEquivalentDate,
    ForecasterRecursive,
    ForecasterRecursiveMultiSeries,
    ForecasterStats,
)
from skforecast.stats import Arima
from skforecast.foundation import FoundationModel, ForecasterFoundation
from skforecast.deep_learning import ForecasterRnn, create_and_compile_model
from skforecast.preprocessing import RollingFeatures, CalendarFeatures
from skforecast.model_selection import (
    TimeSeriesFold,
    backtesting_forecaster,
    backtesting_forecaster_multiseries,
    backtesting_stats,
    backtesting_foundation,
)
from skforecast.metrics import mean_absolute_scaled_error, calculate_coverage
from skforecast.plot import set_dark_theme, plot_prediction_intervals

set_dark_theme()

The backtesting predictions of every model are stored in a dictionary, with the same column names (`pred`, `lower_bound`, `upper_bound` and, for several series, `level`). Two small functions compute the metrics and plot the predictions from that dictionary.

In [ ]:
# Functions to summarize and plot the backtesting predictions
# ==============================================================================
def summarize(predictions, elapsed, series, n_train):
    """
    MAE, MASE, interval coverage and time of each model. MAE and coverage are
    computed over all the predicted values, MASE is the average of the series.
    """
    results = {}
    for model, pred in predictions.items():
        if "level" not in pred.columns:
            pred = pred.assign(level=series.columns[0])
        y_true = pd.concat(
            [series.loc[p.index, level] for level, p in pred.groupby("level")]
        )
        pred = pd.concat([p for _, p in pred.groupby("level")])
        mase = [
            mean_absolute_scaled_error(
                y_true=series.loc[p.index, level],
                y_pred=p["pred"],
                y_train=series[level].iloc[:n_train],
            )
            for level, p in pred.groupby("level")
        ]
        results[model] = {
            "MAE": np.mean(np.abs(y_true.to_numpy() - pred["pred"].to_numpy())),
            "MASE": np.mean(mase),
            "Coverage (80%)": calculate_coverage(
                y_true.to_numpy(),
                pred["lower_bound"].to_numpy(),
                pred["upper_bound"].to_numpy(),
            ),
            "Time (s)": elapsed[model],
        }

    return pd.DataFrame(results).T.sort_values("MASE")


def plot_predictions(predictions, series, level, title):
    """
    One panel per model with its backtesting predictions and interval.
    """
    fig, axes = plt.subplots(2, 3, figsize=(9, 4.5), sharex=True, sharey=True)
    for ax, (model, pred) in zip(axes.flat, predictions.items()):
        if "level" in pred.columns:
            pred = pred.loc[pred["level"] == level]
        plot_prediction_intervals(
            predictions=pred, y_true=series, target_variable=level, title=model, ax=ax
        )
        ax.get_legend().remove()
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
    fig.suptitle(title)
    fig.tight_layout(rect=(0, 0.06, 1, 1))

## Which model wins with exogenous variables?

Daily electricity demand in Victoria (Australia) from 2012 to 2014, with the maximum and mean temperature and the holidays as exogenous variables. The demand has a weekly and a yearly seasonality and depends strongly on the temperature. The last 56 days are predicted in 2 folds of 28 days.

In [ ]:
# Download data and aggregate it to daily values
# ==============================================================================
data = fetch_dataset(name="vic_electricity")
data["Date"] = pd.to_datetime(data["Date"])
data = data.groupby("Date").agg(
    demand=("Demand", "sum"),
    temp_max=("Temperature", "max"),
    temp_mean=("Temperature", "mean"),
    holiday=("Holiday", "max"),
)
data = data.loc["2012-01-01":"2014-12-31"].asfreq("D")
data.index.name = None
data["demand"] = data["demand"] / 1000  # MWh to GWh
data["holiday"] = data["holiday"].astype(float)
data.head(3)

In [ ]:
# Split data into train-test and backtesting partitions
# ==============================================================================
y = data["demand"]
exog = data[["temp_max", "temp_mean", "holiday"]]
n_train = len(data) - 56
print(f"Train dates : {data.index[0].date()} --- {data.index[n_train - 1].date()}  (n={n_train})")
print(f"Test dates  : {data.index[n_train].date()} --- {data.index[-1].date()}  (n={len(data) - n_train})")

cv = TimeSeriesFold(steps=28, initial_train_size=n_train, refit=False, verbose=False)

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(y.iloc[:n_train], label="train")
ax.plot(y.iloc[n_train:], label="test")
ax.set_title("Daily electricity demand (GWh)")
ax.legend();

In [ ]:
# Seasonal naive: the value of the same day of the previous week
# ==============================================================================
predictions, elapsed = {}, {}

start = time.perf_counter()
forecaster = ForecasterEquivalentDate(offset=7)
_, predictions["Seasonal naive"] = backtesting_forecaster(
    forecaster=forecaster, y=y, cv=cv, metric="mean_absolute_error",
    interval=[0.1, 0.9], interval_method="conformal", show_progress=False,
)
elapsed["Seasonal naive"] = time.perf_counter() - start

In [ ]:
# Machine learning: Ridge and LightGBM with lags, rolling mean, calendar features
# and exogenous variables. Intervals use the out-of-sample residuals of a backtest
# over the year before the test period.
# ==============================================================================
cv_calibration = TimeSeriesFold(
    steps=28, initial_train_size=n_train - 365, refit=False, verbose=False
)
estimators = {
    "Ridge": Ridge(),
    "LightGBM": LGBMRegressor(
        n_estimators=400, learning_rate=0.03, num_leaves=15, random_state=123, verbose=-1
    ),
}
for model, estimator in estimators.items():
    start = time.perf_counter()
    forecaster = ForecasterRecursive(
        estimator=estimator,
        lags=7,
        window_features=RollingFeatures(stats="mean", window_sizes=7),
        calendar_features=CalendarFeatures(features=["day_of_week", "day_of_year"]),
        transformer_y=StandardScaler(),
        transformer_exog=StandardScaler(),
    )
    _, pred_calibration = backtesting_forecaster(
        forecaster=forecaster, y=y.iloc[:n_train], exog=exog.iloc[:n_train],
        cv=cv_calibration, metric="mean_absolute_error", show_progress=False,
    )
    forecaster.fit(y=y.iloc[:n_train], exog=exog.iloc[:n_train])
    forecaster.set_out_sample_residuals(
        y_true=y.loc[pred_calibration.index], y_pred=pred_calibration["pred"]
    )
    _, predictions[model] = backtesting_forecaster(
        forecaster=forecaster, y=y, exog=exog, cv=cv, metric="mean_absolute_error",
        interval=[0.1, 0.9], interval_method="conformal",
        use_in_sample_residuals=False, show_progress=False,
    )
    elapsed[model] = time.perf_counter() - start

In [ ]:
# Statistical: AutoARIMA with weekly seasonality and exogenous variables
# ==============================================================================
start = time.perf_counter()
forecaster = ForecasterStats(estimator=Arima(order=None, seasonal_order=None, m=7))
_, predictions["ARIMA"] = backtesting_stats(
    forecaster=forecaster, y=y, exog=exog, cv=cv, metric="mean_absolute_error",
    interval=[0.1, 0.9], show_progress=False, suppress_warnings=True,
)
elapsed["ARIMA"] = time.perf_counter() - start

In [ ]:
# Foundation: Chronos-2 zero-shot, with the exogenous variables as covariates
# ==============================================================================
start = time.perf_counter()
forecaster = ForecasterFoundation(estimator=FoundationModel("autogluon/chronos-2-small"))
_, pred = backtesting_foundation(
    forecaster=forecaster, series=y, exog=exog, cv=cv, metric="mean_absolute_error",
    quantiles=[0.1, 0.5, 0.9], show_progress=False,
)
predictions["Chronos-2"] = pred.rename(
    columns={"q_0.1": "lower_bound", "q_0.5": "pred", "q_0.9": "upper_bound"}
)
elapsed["Chronos-2"] = time.perf_counter() - start

In [ ]:
# Deep learning: LSTM network that predicts the 28 days of each fold at once
# ==============================================================================
start = time.perf_counter()
keras.utils.set_random_seed(123)
network = create_and_compile_model(
    series=data[["demand"]], exog=exog, levels=["demand"], lags=28, steps=28,
    recurrent_layer="LSTM", recurrent_units=32, dense_units=16,
)
forecaster = ForecasterRnn(
    estimator=network, levels=["demand"], lags=28,
    fit_kwargs={"epochs": 50, "batch_size": 32, "verbose": 0},
)
_, predictions["LSTM"] = backtesting_forecaster_multiseries(
    forecaster=forecaster, series=data[["demand"]], exog=exog, cv=cv,
    metric="mean_absolute_error", interval=[0.1, 0.9], show_progress=False,
)
elapsed["LSTM"] = time.perf_counter() - start

In [ ]:
# Results
# ==============================================================================
results_electricity = summarize(predictions, elapsed, data[["demand"]], n_train)
results_electricity.style.format(precision=2).highlight_min(subset=["MASE"], color="green")

In [ ]:
# Backtesting predictions
# ==============================================================================
plot_predictions(predictions, data, level="demand", title="Daily electricity demand (GWh)")

SCENARIO_1_PLACEHOLDER

## What if the series is short?

Monthly expenditure on corticosteroid drugs in Australia (`h2o` dataset): only 204 observations, with trend and yearly seasonality and no exogenous variables. The last 36 months are predicted in 3 folds of 12 months.

In [ ]:
# Download data, split into train-test and backtesting partitions
# ==============================================================================
y = load_demo_dataset()
y.name = "y"
n_train = len(y) - 36
print(f"Train dates : {y.index[0].date()} --- {y.index[n_train - 1].date()}  (n={n_train})")
print(f"Test dates  : {y.index[n_train].date()} --- {y.index[-1].date()}  (n={len(y) - n_train})")

cv = TimeSeriesFold(steps=12, initial_train_size=n_train, refit=False, verbose=False)

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(y.iloc[:n_train], label="train")
ax.plot(y.iloc[n_train:], label="test")
ax.set_title("Monthly expenditure (h2o)")
ax.legend();

In [ ]:
# Seasonal naive: the value of the same month of the previous year
# ==============================================================================
predictions, elapsed = {}, {}

start = time.perf_counter()
forecaster = ForecasterEquivalentDate(offset=12)
_, predictions["Seasonal naive"] = backtesting_forecaster(
    forecaster=forecaster, y=y, cv=cv, metric="mean_absolute_error",
    interval=[0.1, 0.9], interval_method="conformal", show_progress=False,
)
elapsed["Seasonal naive"] = time.perf_counter() - start

In [ ]:
# Machine learning: Ridge and LightGBM with 24 lags and the month. Intervals use
# the out-of-sample residuals of a backtest over the 3 years before the test period.
# ==============================================================================
cv_calibration = TimeSeriesFold(
    steps=12, initial_train_size=n_train - 36, refit=False, verbose=False
)
estimators = {
    "Ridge": Ridge(),
    "LightGBM": LGBMRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=4, random_state=123, verbose=-1
    ),
}
for model, estimator in estimators.items():
    start = time.perf_counter()
    forecaster = ForecasterRecursive(
        estimator=estimator,
        lags=24,
        calendar_features=CalendarFeatures(features=["month"]),
        transformer_y=StandardScaler(),
    )
    _, pred_calibration = backtesting_forecaster(
        forecaster=forecaster, y=y.iloc[:n_train], cv=cv_calibration,
        metric="mean_absolute_error", show_progress=False,
    )
    forecaster.fit(y=y.iloc[:n_train])
    forecaster.set_out_sample_residuals(
        y_true=y.loc[pred_calibration.index], y_pred=pred_calibration["pred"]
    )
    _, predictions[model] = backtesting_forecaster(
        forecaster=forecaster, y=y, cv=cv, metric="mean_absolute_error",
        interval=[0.1, 0.9], interval_method="conformal",
        use_in_sample_residuals=False, show_progress=False,
    )
    elapsed[model] = time.perf_counter() - start

In [ ]:
# Statistical: AutoARIMA with yearly seasonality
# ==============================================================================
start = time.perf_counter()
forecaster = ForecasterStats(estimator=Arima(order=None, seasonal_order=None, m=12))
_, predictions["ARIMA"] = backtesting_stats(
    forecaster=forecaster, y=y, cv=cv, metric="mean_absolute_error",
    interval=[0.1, 0.9], show_progress=False, suppress_warnings=True,
)
elapsed["ARIMA"] = time.perf_counter() - start

In [ ]:
# Foundation: Chronos-2 zero-shot
# ==============================================================================
start = time.perf_counter()
forecaster = ForecasterFoundation(estimator=FoundationModel("autogluon/chronos-2-small"))
_, pred = backtesting_foundation(
    forecaster=forecaster, series=y, cv=cv, metric="mean_absolute_error",
    quantiles=[0.1, 0.5, 0.9], show_progress=False,
)
predictions["Chronos-2"] = pred.rename(
    columns={"q_0.1": "lower_bound", "q_0.5": "pred", "q_0.9": "upper_bound"}
)
elapsed["Chronos-2"] = time.perf_counter() - start

In [ ]:
# Deep learning: LSTM network that predicts the 12 months of each fold at once
# ==============================================================================
start = time.perf_counter()
keras.utils.set_random_seed(123)
network = create_and_compile_model(
    series=y.to_frame(), levels=["y"], lags=24, steps=12,
    recurrent_layer="LSTM", recurrent_units=64, dense_units=16,
)
forecaster = ForecasterRnn(
    estimator=network, levels=["y"], lags=24,
    fit_kwargs={"epochs": 100, "batch_size": 16, "verbose": 0},
)
_, predictions["LSTM"] = backtesting_forecaster_multiseries(
    forecaster=forecaster, series=y.to_frame(), cv=cv,
    metric="mean_absolute_error", interval=[0.1, 0.9], show_progress=False,
)
elapsed["LSTM"] = time.perf_counter() - start

In [ ]:
# Results
# ==============================================================================
results_short = summarize(predictions, elapsed, y.to_frame(), n_train)
results_short.style.format(precision=3).highlight_min(subset=["MASE"], color="green")

In [ ]:
# Backtesting predictions
# ==============================================================================
plot_predictions(predictions, y.to_frame(), level="y", title="Monthly expenditure (h2o)")

SCENARIO_2_PLACEHOLDER

## Many series: one global model or one model per series?

Quarterly overnight trips in Australia from 1998 to 2017, by region and purpose of travel: 304 series of 80 observations. Ridge, LightGBM, Chronos-2 and the LSTM are global models: a single model forecasts all the series. The seasonal naive and ARIMA are local models: one model is fitted for each series. The last 8 quarters are predicted in 2 folds of 4 quarters.

In [ ]:
# Download data, split into train-test and backtesting partitions
# ==============================================================================
data = fetch_dataset(name="australia_tourism")
data["series"] = data["Region"] + " - " + data["Purpose"]
series = data.pivot_table(index=data.index, columns="series", values="Trips").asfreq("QS")
series.index.name = None
series.columns.name = None
n_train = len(series) - 8
print(f"Number of series: {series.shape[1]}")
print(f"Train dates     : {series.index[0].date()} --- {series.index[n_train - 1].date()}  (n={n_train})")
print(f"Test dates      : {series.index[n_train].date()} --- {series.index[-1].date()}  (n={len(series) - n_train})")

cv = TimeSeriesFold(steps=4, initial_train_size=n_train, refit=False, verbose=False)
series.iloc[:3, :4]

In [ ]:
# Local models: one seasonal naive and one AutoARIMA for each series
# ==============================================================================
predictions, elapsed = {}, {}

start = time.perf_counter()
pred_series = []
for level in series.columns:
    _, pred = backtesting_forecaster(
        forecaster=ForecasterEquivalentDate(offset=4), y=series[level], cv=cv,
        metric="mean_absolute_error", interval=[0.1, 0.9],
        interval_method="conformal", show_progress=False,
    )
    pred_series.append(pred.assign(level=level))
predictions["Seasonal naive"] = pd.concat(pred_series)
elapsed["Seasonal naive"] = time.perf_counter() - start

start = time.perf_counter()
pred_series = []
for level in series.columns:
    _, pred = backtesting_stats(
        forecaster=ForecasterStats(estimator=Arima(order=None, seasonal_order=None, m=4)),
        y=series[level], cv=cv, metric="mean_absolute_error", interval=[0.1, 0.9],
        show_progress=False, suppress_warnings=True,
    )
    pred_series.append(pred.assign(level=level))
predictions["ARIMA"] = pd.concat(pred_series)
elapsed["ARIMA"] = time.perf_counter() - start

In [ ]:
# Global machine learning: Ridge and LightGBM trained on the 304 series
# ==============================================================================
start = time.perf_counter()
forecaster = ForecasterRecursiveMultiSeries(
    estimator=Ridge(),
    lags=8,
    window_features=RollingFeatures(stats="mean", window_sizes=4),
    calendar_features=CalendarFeatures(features=["quarter"], encoding="onehot"),
    encoding=None,
    transformer_series=StandardScaler(),
)
_, predictions["Ridge"] = backtesting_forecaster_multiseries(
    forecaster=forecaster, series=series, cv=cv, metric="mean_absolute_error",
    interval=[0.1, 0.9], show_progress=False,
)
elapsed["Ridge"] = time.perf_counter() - start

start = time.perf_counter()
forecaster = ForecasterRecursiveMultiSeries(
    estimator=LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=15, random_state=123, verbose=-1
    ),
    lags=8,
    window_features=RollingFeatures(stats="mean", window_sizes=4),
    calendar_features=CalendarFeatures(features=["quarter"]),
    encoding="ordinal",
)
_, predictions["LightGBM"] = backtesting_forecaster_multiseries(
    forecaster=forecaster, series=series, cv=cv, metric="mean_absolute_error",
    interval=[0.1, 0.9], show_progress=False,
)
elapsed["LightGBM"] = time.perf_counter() - start

In [ ]:
# Foundation: Chronos-2 zero-shot on the 304 series
# ==============================================================================
start = time.perf_counter()
forecaster = ForecasterFoundation(estimator=FoundationModel("autogluon/chronos-2-small"))
_, pred = backtesting_foundation(
    forecaster=forecaster, series=series, cv=cv, metric="mean_absolute_error",
    quantiles=[0.1, 0.5, 0.9], show_progress=False,
)
predictions["Chronos-2"] = pred.rename(
    columns={"q_0.1": "lower_bound", "q_0.5": "pred", "q_0.9": "upper_bound"}
)
elapsed["Chronos-2"] = time.perf_counter() - start

In [ ]:
# Deep learning: one LSTM network that predicts the 304 series at once
# ==============================================================================
start = time.perf_counter()
keras.utils.set_random_seed(123)
network = create_and_compile_model(
    series=series, levels=list(series.columns), lags=8, steps=4,
    recurrent_layer="LSTM", recurrent_units=64, dense_units=64,
)
forecaster = ForecasterRnn(
    estimator=network, levels=list(series.columns), lags=8,
    fit_kwargs={"epochs": 100, "batch_size": 16, "verbose": 0},
)
_, predictions["LSTM"] = backtesting_forecaster_multiseries(
    forecaster=forecaster, series=series, cv=cv,
    metric="mean_absolute_error", interval=[0.1, 0.9], show_progress=False,
)
elapsed["LSTM"] = time.perf_counter() - start

In [ ]:
# Results
# ==============================================================================
results_many = summarize(predictions, elapsed, series, n_train)
results_many.style.format(precision=2).highlight_min(subset=["MASE"], color="green")

In [ ]:
# Backtesting predictions of one of the series
# ==============================================================================
level = "Sydney - Holiday"
plot_predictions(predictions, series, level=level, title=f"Overnight trips: {level}")

SCENARIO_3_PLACEHOLDER

## Summary: accuracy and computation time

The heatmap shows the MASE of each model in the three datasets (lower is better, the best model of each dataset is highlighted). The scatter plot puts the accuracy next to the cost: the time of the whole backtesting, on a logarithmic scale.

In [ ]:
# MASE of each model in each dataset
# ==============================================================================
datasets = {
    "Electricity\n(exogenous)": results_electricity,
    "h2o\n(short series)": results_short,
    "Tourism\n(304 series)": results_many,
}
models = ["Seasonal naive", "Ridge", "LightGBM", "ARIMA", "Chronos-2", "LSTM"]
mase = pd.DataFrame({name: results["MASE"] for name, results in datasets.items()}).loc[models]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.imshow(mase.rank(axis=0).to_numpy(), cmap="viridis_r", aspect="auto")
for (i, j), value in np.ndenumerate(mase.to_numpy()):
    best = value == mase.iloc[:, j].min()
    ax.text(j, i, f"{value:.2f}", ha="center", va="center",
            fontweight="bold" if best else "normal", color="white" if best else "black")
ax.set_xticks(range(mase.shape[1]), mase.columns)
ax.set_yticks(range(mase.shape[0]), mase.index)
ax.grid(False)
ax.set_title("MASE by dataset (color: rank within the dataset)")
fig.tight_layout()

In [ ]:
# Accuracy versus computation time
# ==============================================================================
fig, ax = plt.subplots(figsize=(7, 3.5))
for (name, results), marker in zip(datasets.items(), ["o", "s", "^"]):
    for model in models:
        ax.scatter(results.loc[model, "Time (s)"], results.loc[model, "MASE"],
                   marker=marker, color=f"C{models.index(model)}", s=50)
for model in models:
    ax.scatter([], [], color=f"C{models.index(model)}", label=model)
for (name, results), marker in zip(datasets.items(), ["o", "s", "^"]):
    ax.scatter([], [], color="gray", marker=marker, label=name.replace("\n", " "))
ax.set_xscale("log")
ax.set_xlabel("Backtesting time (seconds, log scale)")
ax.set_ylabel("MASE")
ax.set_title("Accuracy versus computation time")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
fig.tight_layout()

In [ ]:
# Versions and hardware used to measure the times
# ==============================================================================
import lightgbm, torch
print(f"skforecast {skforecast.__version__} | lightgbm {lightgbm.__version__} | "
      f"keras {keras.__version__} | torch {torch.__version__}")
print(f"{platform.processor() or platform.machine()} | {os.cpu_count()} CPU cores | "
      f"GPU available: {torch.cuda.is_available()}")

<div class="admonition note" name="html-admonition" style="background: rgba(255,145,0,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #ff9100; border-color: #ff9100; padding-left: 10px; padding-right: 10px;">

<p class="title">
    <i style="font-size: 18px; color:#ff9100;"></i>
    <b style="color: #ff9100;"> <span style="color: #ff9100;">&#9888;</span> Warning</b>
</p>

These results come from three datasets and one configuration per model; they show trends, not a universal ranking. Keep in mind that:

+ The datasets are public, so they may be part of the data used to pre-train the foundation models, which would make their results optimistic.
+ A machine learning model with good domain features and careful tuning can beat any of these results on a given dataset.
+ Foundation models are slower at prediction time and usually need a GPU in production. The times above were measured on CPU.

Before choosing a model for your data, compare the candidates with [backtesting](backtesting.ipynb) on your own series, as done here.

</div>

## Next steps

+ [Recursive multi-step forecasting](autoregressive-forecaster.ipynb) and [global models](global-forecasting-overview.md) for machine learning models.
+ [Forecasting with foundation models](foundation-forecasting-models.ipynb) to try other models such as TimesFM, Moirai or TabPFN-TS.
+ [Statistical models](forecasting-statistical-models.ipynb): ARIMA, ETS and ARAR.
+ [Deep learning](forecasting-with-deep-learning-rnn-lstm.ipynb) with recurrent neural networks.
+ [Hyperparameter tuning](hyperparameter-tuning-and-lags-selection.ipynb) and [probabilistic forecasting](probabilistic-forecasting-overview.md).